In [ ]:
import sys
sys.path.append('..')

In [2]:
from model.vae import TrackVAE

/export/scratch/ra49veb/conda_envs/pytorch2.6/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
vae = TrackVAE()

Using cache found in /export/home/ra49veb/.cache/torch/hub/facebookresearch_dinov2_main
/export/home/ra49veb/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/export/home/ra49veb/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/export/home/ra49veb/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [4]:
import torch

sd = torch.load(
    "/export/scratch/ra49veb/checkpoints/track-project/243807-repro-n40-wsd/checkpoints/step-80000/model.pt"
)
sd = {k.replace(".mid_level.", "."): v for k, v in sd.items() if "dummy_query_pos" not in k}

vae.load_state_dict(sd)

<All keys matched successfully>

In [5]:
from model.gen import TrackFM

gen = TrackFM(vae=vae)

Using cache found in /export/home/ra49veb/.cache/torch/hub/facebookresearch_dinov2_main


In [7]:
sd = torch.load(
    "/export/scratch/ra49veb/checkpoints/track-project/327613-n16-endt-unlock-repro-actually-rebalanced/checkpoints/step-650000/consolidated.pt"
    # "/export/scratch/ra49veb/checkpoints/track-project/168292-n16-endt-fixed/checkpoints/step-700000/model.pt"
)['train_state']['ema']

sd = {
    k.replace(".mid_level.", ".")
    .replace("unet", "backbone")
    .replace("ae", "vae")
    .replace("backbone.mid_merge", "in_proj")
    .replace("extra_proj", "cond_proj")
    .replace("backbone.mid_split", "out_proj"): v
    for k, v in sd.items()
    if "dummy_query_pos" not in k
}

gen.load_state_dict(sd)

torch.save(sd, '/export/home/ra49veb/dev/track-ae-release/checkpoints/track_gen_ema.pt')

In [8]:
sd = torch.load(
    "/export/scratch/ra49veb/checkpoints/track-project/327613-n16-endt-unlock-repro-actually-rebalanced/checkpoints/step-650000/consolidated.pt"
    # "/export/scratch/ra49veb/checkpoints/track-project/168292-n16-endt-fixed/checkpoints/step-700000/model.pt"
)['model']

sd = {
    k.replace(".mid_level.", ".")
    .replace("unet", "backbone")
    .replace("ae", "vae")
    .replace("backbone.mid_merge", "in_proj")
    .replace("extra_proj", "cond_proj")
    .replace("backbone.mid_split", "out_proj"): v
    for k, v in sd.items()
    if "dummy_query_pos" not in k
}

gen.load_state_dict(sd)

torch.save(sd, '/export/home/ra49veb/dev/track-ae-release/checkpoints/track_gen.pt')

dict_keys(['backbone.3.cross_attn.kv_proj.weight', 'backbone.12.cross_attn.norm_cross.scale', 'backbone.13.self_attn.out_proj.weight', 'backbone.16.self_attn.scale', 'backbone.16.self_attn.norm.linear.weight', 'backbone.16.cross_attn.kv_proj.weight', 'backbone.22.cross_attn.kv_proj.weight', 'vae.encoder.backbone.4.self_attn.scale', 'vae.encoder.backbone.4.self_attn.norm.scale', 'vae.encoder.backbone.4.self_attn.qkv_proj.weight', 'vae.decoder.backbone.1.cross_attn.kv_proj.weight', 'vae.decoder.backbone.7.cross_attn.out_proj.weight', 'vae.decoder.backbone.11.self_attn.pos_emb.yx_freqs', 'vae.decoder.backbone.11.self_attn.pos_emb.t_freqs', 'vae.decoder.backbone.11.cross_attn.scale', 'vae.decoder.backbone.11.cross_attn.norm.scale', 'vae.decoder.backbone.11.cross_attn.norm_cross.scale', 'vae.decoder.backbone.11.cross_attn.q_proj.weight', 'vae.img_embedder.model.blocks.3.mlp.fc2.bias', 'vae.img_embedder.model.blocks.3.ls2.gamma', 'vae.img_embedder.model.blocks.4.norm1.weight', 'vae.img_embed